# Flow matching, SC → every FC condition: learning and predictions (spec v4 E2.1)

Two routes from a subject's SC to their FC in each of the 8 conditions (rest + 7 tasks), each learned by an
**un-anchored flow** (from noise, no ridge) with a **nonlinear** or a **linear** velocity field, one network for all
conditions (a condition token says which one to generate):

- **CCA bridge:** SC PCA → the condition's encoding in E1's 32-dimensional CCA latent → block FC PCA → FC edges;
- **block PCA only:** SC PCA → the condition's 256 block FC PCA codes → FC edges.

Seed 0, retrained at the settings E2.1 selected, with learning logged (`flow_learning.py`). Sections:
(1) setup; (2) learning curves (loss, validation accuracy, validation energy distance) per route and velocity field;
(3) predictions across all 8 conditions for two test subjects, in edge space (deviation from each condition's group
mean); (4) accuracy per condition over all 184 test subjects, flows vs ridge, both routes.

In [ ]:
import sys, json
from pathlib import Path

REPO_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "main.py").exists())
EXP = REPO_ROOT / "scripts" / "experiments" / "joint_flow_matching"
for p in (REPO_ROOT, EXP):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import e21_run as R

SEED = 0
ROUTES = {"cca": "CCA bridge", "pca": "block PCA only"}
MODELS = {"flow_free": "nonlinear velocity", "flow_linear": "linear velocity"}
device = torch.device("cpu")
data = {t: R.Data(SEED, device, target=t) for t in ROUTES}
curves, nets = {}, {}
for t in ROUTES:
    for mdl in MODELS:
        curves[(mdl, t)] = dict(np.load(R.CACHE / "learning" / f"{mdl}_{t}_seed{SEED}.npz"))
        ck = torch.load(R.CACHE / "learning" / f"{mdl}_{t}_seed{SEED}.pt", map_location="cpu")
        net = R.make(mdl, ck["hp"], data[t].d); net.load_state_dict(ck["state"]); net.eval()
        nets[(mdl, t)] = (net, ck["hp"])
refs = {t: json.loads((R.OUT / f"refs_seed{SEED}{'' if t == 'cca' else '_' + t}.json").read_text()) for t in ROUTES}
FC = R.FC
print({k: v[1] for k, v in nets.items()})

import pandas as pd
roi = pd.read_csv(REPO_ROOT / "data" / "atlas_info" / "Glasser_dseg_reformatted.csv")
order_nets = ["visual", "somatosensory", "dorsal attention", "ventral attention", "limbic", "frontoparietal", "default mode"]
lab = roi["community_yeo"].fillna("other").to_numpy()
order = np.concatenate([np.where(lab == n)[0] for n in order_nets + ["other"]])
bounds = np.cumsum([np.sum(lab == n) for n in order_nets + ["other"]])[:-1]
P = len(roi); iu = np.triu_indices(P, k=1)

def to_mat(e):
    M = np.zeros((P, P)); M[iu] = e; M = M + M.T
    return M[np.ix_(order, order)]

def show(ax, e, vmax, title):
    ax.imshow(to_mat(e), cmap="RdBu_r", vmin=-vmax, vmax=vmax, interpolation="nearest")
    for b in bounds:
        ax.axhline(b - 0.5, color="k", lw=0.3); ax.axvline(b - 0.5, color="k", lw=0.3)
    ax.set_xticks([]); ax.set_yticks([]); ax.set_title(title, fontsize=8)

def dr(a, b, mu):
    a, b = a - mu, b - mu
    a, b = a - a.mean(), b - b.mean()
    return float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b)))

## 2. Learning curves

Per route (rows) and quantity (columns): training loss, validation loss (thin) with the smoothed value used for
early stopping (thick), validation accuracy (decoded demeaned r, mean of 16 samples, averaged over the 8 conditions)
and validation energy distance to each subject's 8 encodings. Dashed lines: the closed-form references for that route
(ridge for accuracy, linear-Gaussian for energy). The vertical line marks the epoch whose weights are kept.

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(18, 7.5), layout="constrained")
for i, t in enumerate(ROUTES):
    ref_dr = np.mean([r["demeaned_pearson"] for r in refs[t]["rows"] if r["split"] == "val"])
    ref_en = refs[t]["dist"]["val"]["energy"]
    for mdl, col in zip(MODELS, ("C0", "C1")):
        c = curves[(mdl, t)]
        lab_ = MODELS[mdl]
        axes[i, 0].plot(c["log_epoch"], c["log_train_loss"], color=col, label=lab_)
        axes[i, 1].plot(c["log_epoch"], c["log_val_loss"], color=col, lw=0.6, alpha=0.5)
        axes[i, 1].plot(c["log_epoch"], c["log_val_loss_smooth"], color=col, lw=2, label=lab_)
        axes[i, 2].plot(c["eval_epoch"], c["eval_val_dr"], "o-", ms=3, color=col, label=lab_)
        axes[i, 3].plot(c["eval_epoch"], c["eval_val_energy"], "o-", ms=3, color=col, label=lab_)
        for ax in axes[i]:
            ax.axvline(int(c["best_epoch"]), color=col, ls=":", lw=1)
    axes[i, 2].axhline(ref_dr, color="grey", ls="--", label="ridge (closed form)")
    axes[i, 3].axhline(ref_en, color="grey", ls="--", label="linear-Gaussian")
    axes[i, 0].set_ylabel(f"{ROUTES[t]}\ntraining loss")
    axes[i, 1].set_ylabel("validation loss"); axes[i, 2].set_ylabel("val demeaned r (mean of 8)")
    axes[i, 3].set_ylabel("val energy distance (lower is better)")
for ax in axes[1]:
    ax.set_xlabel("epoch")
for ax in axes[0]:
    ax.legend(frameon=False, fontsize=8)
fig.suptitle(f"Learning of SC → FC (all 8 conditions) by un-anchored flows, seed {SEED}", fontweight="bold")

## 3. Predictions across all 8 conditions (two test subjects, edge space)

Rows: the measured deviation from each condition's group mean, then the generated prediction (mean of 32 samples)
from each route and velocity field, then ridge for comparison. Columns: the 8 conditions. Titles: demeaned r with the
measured FC. Predicted deviations are shown on a narrower colour scale than the measured ones (they are shrunk
toward the group: SC explains a small part of each subject's FC).

In [ ]:
@torch.no_grad()
def predict_all(mdl, t, n_samples=32, split="test"):
    net, hp = nets[(mdl, t)]
    return R.predict(mdl, net, data[t], split, hp["k_sc"], n_samples=n_samples).numpy()     # (8, n, d) raw units

preds = {(mdl, t): predict_all(mdl, t) for t in ROUTES for mdl in MODELS}
sd = data["cca"].L.sd
def edges(t, cond_i, g):
    return data[t].L.decode(cond_i, g)

for subj in (0, 1):
    rows = [("measured", None)] + [((mdl, t), None) for t in ROUTES for mdl in MODELS] + [("ridge (CCA bridge)", "cca")]
    fig, axes = plt.subplots(len(rows), 8, figsize=(18, 2.3 * len(rows)), layout="constrained")
    for j, v in enumerate(FC):
        mu, true = sd.train_mean[v], sd.edges[("test", v)][subj]
        for i, (key, _) in enumerate(rows):
            if key == "measured":
                show(axes[i, j], true - mu, 0.25, v if i == 0 else "")
                continue
            if key == "ridge (CCA bridge)":
                e = edges("cca", j, data["cca"].rc_raw["test"][j][subj:subj + 1])[0]
            else:
                mdl, t = key
                e = edges(t, j, preds[key][j][subj:subj + 1])[0]
            show(axes[i, j], e - mu, 0.08, f"r {dr(e, true, mu):.2f}")
        for i, (key, _) in enumerate(rows):
            name = key if isinstance(key, str) else f"{ROUTES[key[1]]}\n{MODELS[key[0]]}"
            axes[i, 0].set_ylabel(name, fontsize=8)
    fig.suptitle(f"Test subject {subj}: SC → each condition's FC (deviation from the group mean)", fontweight="bold")

## 4. Accuracy per condition (all 184 test subjects)

Test demeaned r per condition for each flow (mean of 32 samples) and the closed-form ridge of each route.

In [ ]:
rows = []
for t in ROUTES:
    for j, v in enumerate(FC):
        mu, true = sd.train_mean[v], sd.edges[("test", v)]
        rp = edges(t, j, data[t].rc_raw["test"][j])
        rows.append(dict(route=ROUTES[t], model="ridge", condition=v,
                         dr=np.mean([dr(rp[i], true[i], mu) for i in range(len(true))])))
        for mdl in MODELS:
            e = edges(t, j, preds[(mdl, t)][j])
            rows.append(dict(route=ROUTES[t], model=f"flow, {MODELS[mdl]}", condition=v,
                             dr=np.mean([dr(e[i], true[i], mu) for i in range(len(true))])))
acc = pd.DataFrame(rows)
tab = acc.pivot_table(index=["route", "model"], columns="condition", values="dr")[FC]
tab["mean of 8"] = tab.mean(1)
display(tab.round(3))

fig, ax = plt.subplots(figsize=(16, 5), layout="constrained")
keys = list(tab.index); w = 0.8 / len(keys); x = np.arange(len(FC))
for k, key in enumerate(keys):
    ax.bar(x + (k - len(keys) / 2 + 0.5) * w, tab.loc[key, FC], w, label=f"{key[0]} · {key[1]}", edgecolor="black", lw=0.6)
ax.set_xticks(x, FC); ax.set_ylabel("test demeaned r (SC → FC)"); ax.legend(frameon=False, fontsize=9, ncol=2)
ax.set_title(f"SC → each FC condition, seed {SEED}: flows vs ridge, CCA bridge vs block PCA only", fontweight="bold")